# 05 — Problemy pochłaniania (problemi di assorbimento)

> Status nauki: ⬜ not-started *(uzupełnij samodzielnie)*

**Źródło:** Abundo, §1.5 „Problemi di assorbimento", s. 16–20 wydruku (PDF 17–21). Obejmuje: Proposizione 1.1, 1.2, 1.2', Przykład 1.10, Esercizi 1.5–1.7.

**Wymagane podstawy:** [04 — klasyfikacja stanów](./04-klasyfikacja-stanow.ipynb) (klasy zamknięte, stany chwilowe), [02 — ruina gracza](./02-ruina-gracza.ipynb) (metoda pierwszego kroku), odwracanie macierzy (notatnik 00).

## Czego się uczymy i po co?

Gdy łańcuch ma klasy zamknięte, prędzej czy później w którejś z nich „utknie". Pytania praktyczne:
1. **Z jakim prawdopodobieństwem** trafi do konkretnej klasy $C$? (probabilità di assorbimento)
2. **Ile razy** po drodze odwiedzi dany stan chwilowy?
3. **Jak długo średnio** potrwa błądzenie, zanim utknie?

To uogólnienie ruiny gracza na dowolne łańcuchy — i jedna z najbardziej „inżynierskich" części teorii.

## 📘 Prawdopodobieństwa pochłonięcia (Proposizione 1.1)

Niech $C$ będzie klasą zamkniętą, a $\lambda_i = P(X_n \in C \text{ dla pewnego } n > 0 \mid X_0 = i)$.

Oczywiste przypadki: $\lambda_i = 1$ dla $i \in C$; $\lambda_i = 0$, gdy $i$ leży w innej klasie zamkniętej $C'$. Ciekawe są tylko **stany chwilowe** $i \in T$.

**Proposizione 1.1.** Dla $i \in T$ liczby $\lambda_i$ są rozwiązaniami (jedynymi, gdy $E$ skończone) układu liniowego:

$$\lambda_i = \sum_{h \in C} p_{ih} + \sum_{j \in T} p_{ij}\,\lambda_j, \qquad i \in T. \tag{1.30}$$

**Intuicja (metoda pierwszego kroku):** pierwszy składnik = wejście do $C$ od razu w jednym kroku; drugi = przejście najpierw do innego stanu chwilowego $j$, skąd pochłonięcie ma prawd. $\lambda_j$.

📘 *Esercizio 1.5*: dla ruiny gracza z $C = \{0\}$ rozwiązaniem układu (1.30) są dokładnie wzory z notatnika 02: $\lambda_i = \frac{(q/p)^a - (q/p)^i}{(q/p)^a - 1}$ ($p \neq q$), $\lambda_i = 1 - i/a$ ($p = q$).

## 📘 Średnia liczba wizyt i macierz fundamentalna (Proposizione 1.2)

Niech $s_{ij}$ = średnia liczba wizyt w stanie chwilowym $j$, gdy startujemy z chwilowego $i$. Wtedy

$$s_{ij} = \delta_{ij} + \sum_{k \in T} p_{ik}\, s_{kj}, \qquad i, j \in T, \tag{1.32}$$

($\delta_{ij}$ — delta Kroneckera: sam start liczy się jako wizyta, gdy $i = j$).

Macierzowo: niech $P_T$ = fragment macierzy $P$ ograniczony do stanów chwilowych ($t \times t$, gdzie $t = |T|$), a $S = (s_{ij})$. Równanie (1.32) to

$$S = I + P_T S \iff (I - P_T)\,S = I \iff S = (I - P_T)^{-1}.$$

Odwracalność $I - P_T$ podręcznik uzasadnia tak: gdyby $X = P_T X$ dla pewnego $X\ne 0$, to $X = (P_T)^n X \to 0$ (bo prawdopodobieństwo pozostania w $T$ przez $n$ kroków maleje do zera) — sprzeczność. Macierz $S$ bywa nazywana **macierzą fundamentalną** łańcucha pochłanianego.

## 📘 Średni czas do pochłonięcia (Proposizione 1.2')

Niech $\eta_i = E(\tau_i)$, gdzie $\tau_i$ to czas spędzony w $T$ przed wejściem do klasy stanów powracających (start $X_0 = i \in T$). Wtedy

$$\eta_i = 1 + \sum_{r \in T} p_{ir}\,\eta_r, \qquad i \in T. \tag{1.32'}$$

Dowód z podręcznika: $\eta_i = \sum_{j\in T} s_{ij}$ (czas w $T$ = suma wizyt we wszystkich stanach chwilowych), sumujemy (1.32) po $j \in T$. Praktycznie: **$\eta_i$ = suma $i$-tego wiersza macierzy $S$**.

📘 *Esercizio 1.6*: dla ruiny gracza układ (1.32') to $\eta_1 = 1 + p\eta_2$, $\eta_i = 1 + q\eta_{i-1} + p\eta_{i+1}$, $\eta_{a-1} = 1 + q\eta_{a-2}$; dla $p=q=\tfrac12$ rozwiązanie $\eta_i = i(a-i)$ — zgadza się z notatnikiem 02.

## 📘 Przykład 1.10 (s. 18 wydruku): ruina gracza, $p = 0.4$, $a = 7$, start $i = 3$

Pytania: (a) średnia liczba chwil, gdy gracz ma 5 euro; (b) gdy ma 2 euro.

Stany chwilowe $T = \{1, \dots, 6\}$, macierz $P_T$ ma $0.4$ nad przekątną i $0.6$ pod nią. Podręcznik odwraca $I - P_T$ **Scilabem** i odczytuje:

$$s_{35} \approx 0.9228, \qquad s_{32} \approx 2.3677.$$

Poniżej odtwarzamy ten rachunek w Pythonie i weryfikujemy symulacją.

In [1]:
import numpy as np

p, a = 0.4, 7
T = list(range(1, a))            # stany chwilowe 1..6
t = len(T)

# macierz P_T: p nad przekątną (i -> i+1), q pod (i -> i-1)
PT = np.zeros((t, t))
for k, i in enumerate(T):
    if i + 1 <= a - 1: PT[k, k + 1] = p
    if i - 1 >= 1:     PT[k, k - 1] = 1 - p

S = np.linalg.inv(np.eye(t) - PT)
print("S = (I - P_T)^(-1):")
print(np.round(S, 4))
print()
print("s_35 =", round(S[T.index(3), T.index(5)], 4), "  (podręcznik: 0.9228)")
print("s_32 =", round(S[T.index(3), T.index(2)], 4), "  (podręcznik: 2.3677)")
print("eta_3 = suma wiersza 3 S =", round(S[T.index(3)].sum(), 4), " (średni czas gry ze startu 3)")

S = (I - P_T)^(-1):
[[1.6149 1.0248 0.6314 0.3691 0.1943 0.0777]
 [1.5372 2.5619 1.5784 0.9228 0.4857 0.1943]
 [1.4206 2.3677 2.999  1.7533 0.9228 0.3691]
 [1.2458 2.0763 2.6299 2.999  1.5784 0.6314]
 [0.9835 1.6391 2.0763 2.3677 2.5619 1.0248]
 [0.5901 0.9835 1.2458 1.4206 1.5372 1.6149]]

s_35 = 0.9228   (podręcznik: 0.9228)
s_32 = 2.3677   (podręcznik: 2.3677)
eta_3 = suma wiersza 3 S = 9.8324  (średni czas gry ze startu 3)


In [2]:
# weryfikacja symulacją Monte Carlo (stałe ziarno)
rng = np.random.default_rng(seed=7)
N = 50_000
wizyty5, wizyty2, czasy = 0, 0, 0
for _ in range(N):
    kap, t_gry = 3, 0
    while 0 < kap < a:
        wizyty5 += (kap == 5)
        wizyty2 += (kap == 2)
        kap += 1 if rng.random() < p else -1
        t_gry += 1
    czasy += t_gry
print(f"symulacja: s_35 ~ {wizyty5/N:.4f}, s_32 ~ {wizyty2/N:.4f}, eta_3 ~ {czasy/N:.4f}")

symulacja: s_35 ~ 0.9104, s_32 ~ 2.3755, eta_3 ~ 9.8482


## 📘 Esercizio 1.7 (s. 19–20 wydruku) — rozwiązany w podręczniku

$E = \{1,2,3,4\}$,

$$P = \begin{pmatrix} 1/2 & 1/2 & 0 & 0 \\ 1/2 & 1/2 & 0 & 0 \\ 1/4 & 1/4 & 1/4 & 1/4 \\ 1/4 & 1/4 & 1/4 & 1/4 \end{pmatrix}$$

(i) Klasyfikacja: $C = \{1,2\}$ klasa zamknięta (powracające), $T = \{3,4\}$ chwilowe.

(ii) Czasy pochłonięcia: układ $\eta_3 = 1 + \tfrac14\eta_3 + \tfrac14\eta_4$, $\eta_4 = 1 + \tfrac14\eta_3 + \tfrac14\eta_4$. Z symetrii $\eta_3 = \eta_4 = \eta$: $\eta = 1 + \tfrac12\eta \Rightarrow \eta = 2$.

Prawdopodobieństwa pochłonięcia: $\lambda_3 = \tfrac12 + \tfrac14\lambda_3 + \tfrac14\lambda_4$ itd., rozwiązanie $\lambda_3 = \lambda_4 = 1$ — zgodnie z oczekiwaniem (jedyna klasa zamknięta musi w końcu wchłonąć łańcuch).

In [3]:
import numpy as np

# Esercizio 1.7 w kodzie
P = np.array([[1/2, 1/2, 0,   0  ],
              [1/2, 1/2, 0,   0  ],
              [1/4, 1/4, 1/4, 1/4],
              [1/4, 1/4, 1/4, 1/4]])
T = [2, 3]                       # indeksy 0-bazowe stanów 3, 4
C = [0, 1]

PT = P[np.ix_(T, T)]
S = np.linalg.inv(np.eye(2) - PT)
eta = S.sum(axis=1)
print("eta_3, eta_4 =", eta, " (podręcznik: 2, 2)")

# lambda_i: (I-PT) lambda = wektor [sum_{h in C} p_ih]
b = P[np.ix_(T, C)].sum(axis=1)
lam = np.linalg.solve(np.eye(2) - PT, b)
print("lambda_3, lambda_4 =", lam, " (podręcznik: 1, 1)")

eta_3, eta_4 = [2. 2.]  (podręcznik: 2, 2)
lambda_3, lambda_4 = [1. 1.]  (podręcznik: 1, 1)


## Typowe błędy

1. Układ (1.30) piszemy **tylko dla stanów chwilowych** — dla $i \in C$ mamy $\lambda_i = 1$ z definicji, wstawianie ich do układu psuje rachunek.
2. W (1.32) łatwo zgubić $\delta_{ij}$ — sam start liczy się jako wizyta.
3. $P_T$ to **wycinek** macierzy $P$ (wiersze i kolumny ze zbioru $T$), nie cała macierz; jej wiersze zwykle NIE sumują się do 1 (część masy „ucieka" do klas zamkniętych) — i właśnie dlatego $I - P_T$ jest odwracalna.
4. $\eta_i$ liczy kroki **do wejścia** do klasy powracającej; czas w samej klasie się nie liczy.

## Podsumowanie

- Prawdopodobieństwa pochłonięcia: układ $\lambda_i = \sum_{h\in C} p_{ih} + \sum_{j\in T} p_{ij}\lambda_j$.
- Średnie liczby wizyt: $S = (I - P_T)^{-1}$ (macierz fundamentalna).
- Średni czas do pochłonięcia: $\eta_i = $ suma $i$-tego wiersza $S$, równoważnie $\eta_i = 1 + \sum_r p_{ir}\eta_r$.

## Pytania kontrolne

1. Dlaczego $I - P_T$ jest odwracalna dla skończonego łańcucha?
2. W Esercizio 1.7: jak zmieniłby się $\eta_3$, gdyby $p_{33} = 1/2$, $p_{34} = 0$ (reszta wiersza bez zmian: $p_{31}=p_{32}=1/4$)?
3. Ile wynosi suma $\lambda_i$ po wszystkich klasach zamkniętych, dla ustalonego stanu chwilowego $i$ w skończonym łańcuchu?

*(Odpowiedzi poniżej.)*

## Odpowiedzi

1. Bo $(P_T)^n \to 0$: prawdopodobieństwo pozostania w stanach chwilowych przez $n$ kroków dąży do zera, więc jedynym rozwiązaniem $X = P_T X$ jest $X=0$, czyli $\det(I - P_T) \ne 0$.
2. Układ: $\eta_3 = 1 + \tfrac12\eta_3 \Rightarrow \eta_3 = 2$ — bez zmian! (Ale to przypadek; zmieniło się tylko to, że z 3 nie chodzimy już przez 4.) Dokładniej: $\eta_4 = 1 + \tfrac14\eta_3 + \tfrac14\eta_4 \Rightarrow \eta_4 = 2$ również.
3. Równa 1: łańcuch skończony musi z prawdopodobieństwem 1 trafić do którejś klasy zamkniętej, a zdarzenia „pochłonięcie w $C_k$" są rozłączne.

## Mini słowniczek IT–PL

| włoski | polski |
|---|---|
| problemi di assorbimento | problemy pochłaniania |
| probabilità di assorbimento | prawdopodobieństwo pochłonięcia |
| numero medio di visite | średnia liczba wizyt |
| tempo medio di assorbimento | średni czas pochłonięcia |
| matrice identità (Id) | macierz jednostkowa |
| sistema lineare | układ równań liniowych |
| matrice non singolare | macierz nieosobliwa (odwracalna) |